# TF-SED: Time-Frequency Sound-Event Box Generation

Generate weak **2-D time-frequency bounding boxes** over [AudioSet](https://research.google.com/audioset/)
using [`Qwen/Qwen3-Omni-30B-A3B-Instruct`](https://huggingface.co/Qwen/Qwen3-Omni-30B-A3B-Instruct).

**Pipeline per clip**

1. Slice the 10 s clip into overlapping 5 s windows (2.5 s hop).
2. For each window, give the model *both* a gridded 0–4 kHz spectrogram image
   and the raw 16 kHz audio, conditioned on the clip's AudioSet labels, and ask
   for a JSON array of boxes (`label, onset_s, offset_s, freq_low_hz, freq_high_hz`).
3. **Snap** each box to real STFT energy (percentile threshold + morphology) so
   the bounds track the signal rather than the raw model estimate.
4. Map labels to the official 527 AudioSet classes (inline — no retroactive pass).
5. Merge boxes across windows with per-label NMS.

Output: one JSON per clip, plus a flat `metadata.csv` manifest.
Labels are **weak** (model-generated, not human-verified).

> Requires a CUDA GPU with enough memory for the 30B model (tested on a single H100).


In [ ]:
# Dependencies (uncomment to install)
# !pip install -q transformers accelerate torch librosa soundfile datasets \
#     pillow scipy matplotlib pydantic tqdm
# !pip install -q flash-attn --no-build-isolation   # optional, for the H100 speedup


In [ ]:
import io
import re
import json
import time
import warnings
from pathlib import Path

import numpy as np
import librosa
import scipy.ndimage as ndimage
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from datasets import load_dataset
from IPython.display import Audio, display
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForTextToWaveform
from transformers import logging as hf_logging

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
hf_logging.set_verbosity_error()


## 1. Configuration

In [ ]:
MODEL_ID    = "Qwen/Qwen3-Omni-30B-A3B-Instruct"
AUDIO_SR    = 16_000      # model sample rate
FMAX        = 4_000       # analysis ceiling (Hz) — spectrogram + boxes
WINDOW_S    = 5.0         # sliding-window length
HOP_S       = 2.5         # hop (50% overlap)
IOU_THRESH  = 0.3         # NMS merge threshold for boxes across windows
N_FFT       = 1024
HOP_LENGTH  = 256

MAX_CLIPS_PER_LABEL = 3000   # cap how many clips are processed per AudioSet label
LIMIT_CLIPS         = None    # set to an int for a quick smoke run; None = no limit

OUTPUT_DIR = Path("tfsed_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Load the model

In [ ]:
print("Loading Qwen3-Omni ...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
engine = AutoModelForTextToWaveform.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="flash_attention_2",   # drop this line if flash-attn isn't installed
)
if hasattr(engine, "disable_talker"):
    engine.disable_talker()        # we only need the text head
engine.eval()
print("Model ready.")


## 3. AudioSet label vocabulary & snapping

Model output labels are mapped to the official 527 AudioSet classes. Policy
(precision over recall — drop rather than guess):

1. case-insensitive exact match;
2. a small curated override table (handles the common comma-form labels);
3. conservative fuzzy match (cutoff 0.8); otherwise the box is dropped.

The override table also fixes a few known false-friends the raw fuzzy matcher
got wrong (e.g. `Thud→Thunder`, `Chewing→Cheering`, `Crying→Crushing`).


In [ ]:
import urllib.request, csv as _csv
from difflib import get_close_matches

# Official 527 classes. Canonical source is the AudioSet bucket; the GitHub
# mirror below is identical and more reliable to fetch.
_LABELS_URL = ("https://raw.githubusercontent.com/qiuqiangkong/"
               "audioset_tagging_cnn/master/metadata/class_labels_indices.csv")
_rows = list(_csv.DictReader(io.StringIO(
    urllib.request.urlopen(_LABELS_URL).read().decode("utf-8"))))
AUDIOSET_LABELS    = sorted({r["display_name"] for r in _rows})
AUDIOSET_LABEL_SET = set(AUDIOSET_LABELS)
_LOWER_TO_CANON    = {name.lower(): name for name in AUDIOSET_LABELS}

# Curated overrides (keys are matched case-insensitively). None == drop the box.
_MANUAL_SNAP = {
    "gunshot": "Gunshot, gunfire",
    "cattle": "Cattle, bovinae",
    "violin": "Violin, fiddle", "fiddle": "Violin, fiddle",
    "chicken": "Chicken, rooster",
    "crowing": "Crowing, cock-a-doodle-doo",
    "crowing - cock-a-doodle-doo": "Crowing, cock-a-doodle-doo",
    "boat": "Boat, Water vehicle",
    "whoosh": "Whoosh, swoosh, swish",
    "smash": "Smash, crash", "whack": "Smash, crash",
    "dtmf": "Telephone",
    "air horn": "Air horn, truck horn",
    "fire engine": "Fire engine, fire truck (siren)",
    "burst": "Explosion",
    "dental drill": "Drill", "dentist's drill": "Drill",
    "motorboat": "Motorboat, speedboat",
    "jingle": "Jingle bell",
    "burping": "Burping, eructation, belching",
    "child speech": "Child speech, kid speaking",
    "fixed-wing aircraft": "Fixed-wing aircraft, airplane",
    "ratchet": "Ratchet, pawl",
    "electric shaver": "Electric shaver, electric razor",
    "wail": "Wail, moan", "moan": "Wail, moan",
    "neigh": "Neigh, whinny",
    "beep": "Beep, bleep", "bleep": "Beep, bleep",
    "marimba": "Marimba, xylophone",
    "vehicle horn": "Vehicle horn, car horn, honking",
    "crying": "Crying, sobbing", "sobbing": "Crying, sobbing",
    "splash": "Splash, splatter", "splatter": "Splash, splatter",
    "crumpling": "Crumpling, crinkling",
    "chewing": "Chewing, mastication",
    "propeller": "Propeller, airscrew",
    "wind instrument": "Wind instrument, woodwind instrument",
    "woodwind instrument": "Wind instrument, woodwind instrument",
    # ambiguous / no clean official class -> drop
    "walk": None, "fly": None, "thump": None, "thud": None,
    "chuckle": None, "pop": None, "smoke detector": None,
}

def snap_to_audioset(label):
    """Map a free-form label to an official AudioSet class, or None to drop it."""
    if not label:
        return None
    key = str(label).strip()
    canon = _LOWER_TO_CANON.get(key.lower())          # 1. exact (case-insensitive)
    if canon:
        return canon
    if key.lower() in _MANUAL_SNAP:                   # 2. curated overrides
        return _MANUAL_SNAP[key.lower()]
    hits = get_close_matches(key, AUDIOSET_LABELS, n=1, cutoff=0.8)  # 3. fuzzy
    return hits[0] if hits else None

print(f"Loaded {len(AUDIOSET_LABELS)} AudioSet labels.")
for t in ["speech", "Thud", "Chicken", "Walk", "Neigh"]:
    print(f"  snap({t!r}) -> {snap_to_audioset(t)!r}")


## 4. Spectrogram rendering & prompt

In [ ]:
TF_SYSTEM_PROMPT = (
    "You localize sound events jointly in time and frequency. "
    "Read the spectrogram and listen to the audio, then report boxes precisely."
)

def build_user_prompt(labels):
    label_txt = ", ".join(labels) if labels else "unspecified events"
    return (
        "=== KNOWN CONTEXT ===\n"
        f"The clip contains: [{label_txt}].\n"
        "You are analyzing a 5 s slice — only annotate sounds you actually hear. "
        "Do not guess.\n\n"
        "=== TASK ===\n"
        "Output a JSON array of bounding boxes. Fields: "
        "label, onset_s, offset_s, freq_low_hz, freq_high_hz, "
        "band, spectral_character, time_evolution."
    )

def make_spectrogram_image(wav, sr):
    """Gridded magma spectrogram (0..FMAX) used as the visual input."""
    S = np.abs(librosa.stft(wav, n_fft=N_FFT, hop_length=HOP_LENGTH)) ** 2
    S_db = librosa.power_to_db(S, ref=np.max)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    keep = freqs <= FMAX
    S_db = S_db[keep]
    f_top = float(freqs[keep][-1]) if keep.any() else sr / 2

    fig, ax = plt.subplots(figsize=(6, 4), dpi=110)
    ax.imshow(S_db, aspect="auto", origin="lower", cmap="magma",
              extent=[0, len(wav) / sr, 0, f_top])
    ax.set_xticks(np.arange(0, len(wav) / sr + 0.5, 0.5))
    ax.set_yticks(np.arange(0, f_top + 500, 500))
    ax.grid(color="white", linestyle="-", linewidth=1, alpha=0.8)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


## 5. Box schema & JSON parsing

In [ ]:
from pydantic import BaseModel, RootModel, Field
from typing import List

class TFBox(BaseModel):
    label:              str
    onset_s:            float = Field(..., ge=0.0)
    offset_s:           float = Field(..., ge=0.0)
    freq_low_hz:        float = Field(..., ge=0.0)
    freq_high_hz:       float = Field(..., ge=0.0)
    band:               str = ""
    spectral_character: str = ""
    time_evolution:     str = ""

def extract_json_payload(text: str) -> str:
    """Strip markdown fences and isolate the outermost JSON array/object."""
    text = text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.S | re.I)
    if fenced:
        text = fenced.group(1).strip()
    starts = [i for i in (text.find("["), text.find("{")) if i != -1]
    if starts:
        start = min(starts)
        end = max(text.rfind("]"), text.rfind("}"))
        if end > start:
            text = text[start:end + 1]
    return text


## 6. Energy refinement & NMS

`refine_box` snaps a rough model box to the high-energy region of the actual
STFT around it. `nms_merge` merges overlapping same-label boxes coming from
adjacent windows.


In [ ]:
def refine_box(wav, sr, t0, t1, f0, f1, duration, char=""):
    """Tighten a box onto real spectral energy (per-character thresholding)."""
    S = np.abs(librosa.stft(wav, n_fft=N_FFT, hop_length=HOP_LENGTH)) ** 2
    S_db = librosa.power_to_db(S, ref=np.max)
    times = librosa.frames_to_time(np.arange(S_db.shape[1]), sr=sr, hop_length=HOP_LENGTH)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)

    dt = times[1] - times[0] if len(times) > 1 else max(duration / 200, 1e-3)
    t_mask = (times >= max(0.0, t0 - 0.35)) & (times <= min(duration, t1 + 0.35))
    f_mask = (freqs >= max(0.0, f0 - 250.0)) & (freqs <= min(FMAX, f1 + 250.0))
    if not t_mask.any() or not f_mask.any():
        return t0, t1, f0, f1

    t_idx, f_idx = np.where(t_mask)[0], np.where(f_mask)[0]
    region = S_db[np.ix_(f_idx, t_idx)]
    if region.size == 0:
        return t0, t1, f0, f1

    region_norm = region - np.percentile(region, 30)
    char_l = str(char).lower()
    if "noisy" in char_l or "broadband" in char_l:
        mask = ndimage.binary_closing(region_norm >= np.percentile(region_norm, 80),
                                      structure=np.ones((5, 5)))
    elif "impulsive" in char_l:
        mask = ndimage.binary_closing(region_norm >= np.percentile(region_norm, 85),
                                      structure=np.ones((7, 2)))
    else:
        p_hi = np.percentile(region_norm, 97)
        mask = region_norm >= (p_hi if p_hi > 0 else np.percentile(region_norm, 95))

    if mask.sum() < 8:
        return t0, t1, f0, f1

    rows, cols = np.where(mask)
    c0, c1 = int(np.floor(np.quantile(cols, 0.03))), int(np.ceil(np.quantile(cols, 0.97)))
    r0, r1 = int(np.floor(np.quantile(rows, 0.05))), int(np.ceil(np.quantile(rows, 0.95)))
    c0, c1 = max(0, min(c0, len(t_idx) - 1)), max(0, min(c1, len(t_idx) - 1))
    r0, r1 = max(0, min(r0, len(f_idx) - 1)), max(0, min(r1, len(f_idx) - 1))
    return (float(times[t_idx[c0]]), float(min(duration, times[t_idx[c1]] + dt)),
            float(freqs[f_idx[r0]]), float(freqs[f_idx[r1]]))

def box_iou(b1, b2):
    """IoU of two (t0, t1, f0, f1) boxes."""
    it0, it1 = max(b1[0], b2[0]), min(b1[1], b2[1])
    if0, if1 = max(b1[2], b2[2]), min(b1[3], b2[3])
    if it1 <= it0 or if1 <= if0:
        return 0.0
    inter = (it1 - it0) * (if1 - if0)
    a1 = (b1[1] - b1[0]) * (b1[3] - b1[2])
    a2 = (b2[1] - b2[0]) * (b2[3] - b2[2])
    return inter / float(a1 + a2 - inter)

def nms_merge(events, iou_threshold=IOU_THRESH):
    """Merge overlapping same-label boxes (union of bounds)."""
    merged, groups = [], {}
    for ev in events:
        groups.setdefault(ev.get("label", "unknown"), []).append(ev)
    for _, group in groups.items():
        while group:
            cur = group.pop(0)
            box_c = [cur["onset_s"], cur["offset_s"], cur["freq_low_hz"], cur["freq_high_hz"]]
            i = 0
            while i < len(group):
                cand = group[i]
                box_k = [cand["onset_s"], cand["offset_s"], cand["freq_low_hz"], cand["freq_high_hz"]]
                if box_iou(box_c, box_k) > iou_threshold:
                    cur["onset_s"] = min(cur["onset_s"], cand["onset_s"])
                    cur["offset_s"] = max(cur["offset_s"], cand["offset_s"])
                    cur["freq_low_hz"] = min(cur["freq_low_hz"], cand["freq_low_hz"])
                    cur["freq_high_hz"] = max(cur["freq_high_hz"], cand["freq_high_hz"])
                    box_c = [cur["onset_s"], cur["offset_s"], cur["freq_low_hz"], cur["freq_high_hz"]]
                    group.pop(i)
                else:
                    i += 1
            merged.append(cur)
    return merged


## 7. Per-clip inference

In [ ]:
def process_clip(wav, sr, labels):
    """Run windowed inference over one clip and return merged, label-snapped boxes."""
    duration = len(wav) / sr
    chunks = []
    for start_s in np.arange(0, duration, HOP_S):
        end_s = min(start_s + WINDOW_S, duration)
        if end_s - start_s < 1.0:
            continue
        chunks.append({"start_s": start_s, "wav": wav[int(start_s * sr):int(end_s * sr)]})
    if not chunks:
        return []

    texts, images, audios = [], [], []
    for ch in chunks:
        img = make_spectrogram_image(ch["wav"], sr)
        conv = [
            {"role": "system", "content": [{"type": "text", "text": TF_SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "audio", "audio": (ch["wav"], sr)},
                {"type": "text", "text": build_user_prompt(labels)},
            ]},
        ]
        texts.append(processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False))
        images.append(img)
        audios.append(ch["wav"])

    inputs = processor(text=texts, audio=audios, images=images,
                       return_tensors="pt", padding=True).to(engine.device, engine.dtype)
    out = engine.generate(**inputs, max_new_tokens=1024,
                          pad_token_id=processor.tokenizer.eos_token_id)
    replies = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                     skip_special_tokens=True)

    events = []
    for ch, reply in zip(chunks, replies):
        reply = reply.strip()
        if not reply:
            continue
        try:
            boxes = RootModel[List[TFBox]].model_validate_json(extract_json_payload(reply)).root
        except Exception:
            continue
        for box in boxes:
            snapped = snap_to_audioset(box.label)   # inline label snapping
            if snapped is None:
                continue
            t0, t1, f0, f1 = refine_box(ch["wav"], sr, box.onset_s, box.offset_s,
                                        box.freq_low_hz, box.freq_high_hz,
                                        len(ch["wav"]) / sr, box.spectral_character)
            ev = box.model_dump()
            ev.update({
                "label": snapped,
                "onset_s": round(t0 + ch["start_s"], 3),
                "offset_s": round(t1 + ch["start_s"], 3),
                "freq_low_hz": round(f0, 1),
                "freq_high_hz": round(f1, 1),
            })
            # drop degenerate boxes
            if ev["offset_s"] > ev["onset_s"] and ev["freq_high_hz"] > ev["freq_low_hz"]:
                events.append(ev)
    return nms_merge(events)


## 8. Sanity check on one clip

In [ ]:
def visualize(wav, sr, events, title="TF-SED predictions"):
    display(Audio(wav, rate=sr))
    S_db = librosa.power_to_db(
        np.abs(librosa.stft(wav, n_fft=N_FFT, hop_length=HOP_LENGTH)) ** 2, ref=np.max)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    keep = freqs <= FMAX
    S_db = S_db[keep]
    f_top = float(freqs[keep][-1]) if keep.any() else sr / 2
    duration = len(wav) / sr

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(S_db, aspect="auto", origin="lower", cmap="magma",
              extent=[0, duration, 0, f_top])
    for i, ev in enumerate(events):
        col = plt.cm.tab10(i % 10)
        ax.add_patch(patches.Rectangle(
            (ev["onset_s"], ev["freq_low_hz"]),
            max(0.01, ev["offset_s"] - ev["onset_s"]),
            max(10, ev["freq_high_hz"] - ev["freq_low_hz"]),
            linewidth=2.5, edgecolor=col, facecolor="none"))
        ax.text(ev["onset_s"] + 0.02, min(f_top - 50, ev["freq_high_hz"]),
                ev["label"], color="white", fontsize=10, weight="bold", va="top",
                bbox=dict(facecolor=col, alpha=0.75, edgecolor="none", boxstyle="round,pad=0.3"))
    ax.set_xlim(0, duration); ax.set_ylim(0, f_top)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Frequency (Hz)"); ax.set_title(title)
    plt.tight_layout(); plt.show()

def parse_labels(clip):
    raw = clip.get("human_labels", [])
    if isinstance(raw, str):
        return [l.strip() for l in raw.split(",") if l.strip()]
    if hasattr(raw, "__iter__"):
        return sorted({str(l) for l in raw})
    return []

def load_audio(clip):
    wav = np.asarray(clip["audio"]["array"], dtype=np.float32)
    sr = clip["audio"]["sampling_rate"]
    if wav.ndim > 1:
        wav = wav.mean(axis=0)
    if sr != AUDIO_SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=AUDIO_SR)
    return wav, AUDIO_SR

# one random clip
for clip in load_dataset("agkphysics/AudioSet", split="train",
                         streaming=True).shuffle(buffer_size=100).take(1):
    wav, sr = load_audio(clip)
    labels = parse_labels(clip)
    print("Ground-truth labels:", labels)
    visualize(wav, sr, process_clip(wav, sr, labels), title=f"Sanity check: {labels}")


## 9. Full generation loop

Streams AudioSet, caps clips per label, and writes one JSON per clip. Safe to
re-run — already-written clips are skipped.


In [ ]:
stream = load_dataset("agkphysics/AudioSet", split="train", streaming=True)
label_counts = {}
n_written = 0

for clip_idx, clip in enumerate(tqdm(stream, desc="Generating")):
    if LIMIT_CLIPS is not None and n_written >= LIMIT_CLIPS:
        break

    labels = parse_labels(clip)
    if not labels:
        continue
    if any(label_counts.get(l, 0) >= MAX_CLIPS_PER_LABEL for l in labels):
        continue

    out_path = OUTPUT_DIR / f"clip_{clip_idx:06d}.json"
    if out_path.exists():
        for l in labels:
            label_counts[l] = label_counts.get(l, 0) + 1
        continue

    wav, sr = load_audio(clip)
    boxes = process_clip(wav, sr, labels)
    out_path.write_text(json.dumps(
        {"clip_idx": clip_idx, "dataset_labels": labels, "weak_labels": boxes},
        indent=2), encoding="utf-8")

    for l in labels:
        label_counts[l] = label_counts.get(l, 0) + 1
    n_written += 1

print(f"Done. Wrote {n_written} clips to {OUTPUT_DIR}/")


## 10. Export the Hugging Face manifest

Builds `metadata.csv` (one row per clip) from the JSON files. The `weak_labels`
are also embedded as `weak_labels_json` so the manifest is self-contained.


In [ ]:
import csv

json_files = sorted(OUTPUT_DIR.glob("clip_*.json"))
rows = []
for jf in json_files:
    rec = json.loads(jf.read_text(encoding="utf-8"))
    labels = sorted({str(l) for l in rec.get("dataset_labels", []) if str(l).strip()})
    boxes = rec.get("weak_labels", [])
    rows.append({
        "clip_idx": rec.get("clip_idx", jf.stem),
        "json_relpath": f"jsons/{jf.name}",
        "dataset_labels": "|".join(labels),
        "num_labels": len(labels),
        "num_boxes": len(boxes),
        "weak_labels_json": json.dumps(boxes, ensure_ascii=False),
    })

csv_path = OUTPUT_DIR / "metadata.csv"
with csv_path.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["clip_idx", "json_relpath", "dataset_labels",
                                      "num_labels", "num_boxes", "weak_labels_json"])
    w.writeheader()
    w.writerows(rows)

print(f"Wrote {csv_path} with {len(rows)} clips, {sum(r['num_boxes'] for r in rows)} boxes.")
